# View a LAMMPS trajectory

This notebook loads one of the `*.lammpstrj` files under `scripts/LAMMPs/MD/runs`, maps the numeric LAMMPS atom types back to chemical symbols using the companion `*.xyz` structure file, and then offers a few notebook-friendly viewing modes.

Use the `nglview` section for the smoothest playback and scrubbing, and the optional `py3Dmol` section for a prettier Materials Project-like static/step-through view.

Change `RUN_NAME`, `STRIDE`, or `MAX_FRAMES` in the next cell to inspect a different run.


In [ ]:
from pathlib import Path

EXAMPLES = {
    "BaTiO3 hysteresis (direct-trained)": {
        "run_dir": Path("MD/runs/BaTiO3-mp-5986-sc3x3x3-0K-5GHz-hysteresis-2026-05-01_214723/BaTiO3-mp-5986"),
        "trajectory": "hysteresis.lammpstrj",
        "reference": "BaTiO3-mp-5986.xyz",
    },
    "BaTiO3 hysteresis (foundation)": {
        "run_dir": Path("MD/runs/BaTiO3-mp-5986-sc2x2x2-0K-5GHz-hysteresis-2026-05-01_142623/BaTiO3-mp-5986"),
        "trajectory": "hysteresis.lammpstrj",
        "reference": "BaTiO3-mp-5986.xyz",
    },
    "SiO2 production (foundation)": {
        "run_dir": Path("MD/runs/SiO2-mp-7000-sc1x1x1-300K-200ps-2026-05-03_101733/SiO2-mp-7000"),
        "trajectory": "production.lammpstrj",
        "reference": "SiO2-mp-7000.xyz",
    },
}

RUN_NAME = "SiO2 production (foundation)"
STRIDE = 100
MAX_FRAMES = 1000
WRAP_POSITIONS = False

selection = EXAMPLES[RUN_NAME]
RUN_DIR = selection["run_dir"]
TRAJ_PATH = RUN_DIR / selection["trajectory"]
REF_PATH = RUN_DIR / selection["reference"]

print(f"Run: {RUN_NAME}")
print(f"Trajectory: {TRAJ_PATH}")
print(f"Reference structure: {REF_PATH}")

Run: BaTiO3 hysteresis (direct-trained)
Trajectory: MD/runs/BaTiO3-mp-5986-sc3x3x3-0K-5GHz-hysteresis-2026-05-01_214723/BaTiO3-mp-5986/hysteresis.lammpstrj
Reference structure: MD/runs/BaTiO3-mp-5986-sc3x3x3-0K-5GHz-hysteresis-2026-05-01_214723/BaTiO3-mp-5986/BaTiO3-mp-5986.xyz


In [7]:
import numpy as np
from ase import Atoms
from ase.io import read


def _parse_box(box_lines, triclinic):
    if triclinic:
        xlo_bound, xhi_bound, xy = map(float, box_lines[0].split())
        ylo_bound, yhi_bound, xz = map(float, box_lines[1].split())
        zlo_bound, zhi_bound, yz = map(float, box_lines[2].split())

        xlo = xlo_bound - min(0.0, xy, xz, xy + xz)
        xhi = xhi_bound - max(0.0, xy, xz, xy + xz)
        ylo = ylo_bound - min(0.0, yz)
        yhi = yhi_bound - max(0.0, yz)
        zlo = zlo_bound
        zhi = zhi_bound

        return np.array([
            [xhi - xlo, 0.0, 0.0],
            [xy, yhi - ylo, 0.0],
            [xz, yz, zhi - zlo],
        ])

    xlo, xhi = map(float, box_lines[0].split())
    ylo, yhi = map(float, box_lines[1].split())
    zlo, zhi = map(float, box_lines[2].split())
    return np.array([
        [xhi - xlo, 0.0, 0.0],
        [0.0, yhi - ylo, 0.0],
        [0.0, 0.0, zhi - zlo],
    ])


def iter_lammpstrj_frames(path):
    with open(path) as handle:
        while True:
            line = handle.readline()
            if not line:
                return
            if not line.startswith("ITEM: TIMESTEP"):
                raise RuntimeError(f"Unexpected line while parsing {path}: {line!r}")

            timestep = int(handle.readline().strip())

            number_header = handle.readline().strip()
            if not number_header.startswith("ITEM: NUMBER OF ATOMS"):
                raise RuntimeError(f"Unexpected atom-count header: {number_header!r}")
            natoms = int(handle.readline().strip())

            box_header = handle.readline().strip()
            triclinic = "xy xz yz" in box_header
            cell = _parse_box([handle.readline(), handle.readline(), handle.readline()], triclinic)

            atom_header = handle.readline().strip().split()[2:]
            rows = [handle.readline().split() for _ in range(natoms)]
            columns = {
                name: np.array([float(row[i]) for row in rows])
                for i, name in enumerate(atom_header)
            }

            ids = columns["id"].astype(int)
            types = columns["type"].astype(int)

            position_columns = None
            for candidate in (("xu", "yu", "zu"), ("x", "y", "z"), ("xs", "ys", "zs")):
                if all(name in columns for name in candidate):
                    position_columns = candidate
                    break
            if position_columns is None:
                raise RuntimeError(f"Could not find coordinates in columns: {atom_header}")

            positions = np.column_stack([columns[name] for name in position_columns])
            forces = None
            if all(name in columns for name in ("fx", "fy", "fz")):
                forces = np.column_stack([columns["fx"], columns["fy"], columns["fz"]])

            yield {
                "timestep": timestep,
                "ids": ids,
                "types": types,
                "positions": positions,
                "forces": forces,
                "cell": cell,
            }


def infer_type_map(traj_path, ref_xyz_path):
    reference = read(ref_xyz_path)
    first_frame = next(iter_lammpstrj_frames(traj_path))
    order = np.argsort(first_frame["ids"])

    mapping_sets = {}
    for atom_id, atom_type in zip(first_frame["ids"][order], first_frame["types"][order]):
        symbol = reference[int(atom_id) - 1].symbol
        mapping_sets.setdefault(int(atom_type), set()).add(symbol)

    bad = {key: value for key, value in mapping_sets.items() if len(value) != 1}
    if bad:
        raise RuntimeError(f"Could not infer a unique type map: {bad}")

    return {key: next(iter(value)) for key, value in mapping_sets.items()}


def load_frames(traj_path, ref_xyz_path, stride=1, max_frames=None, wrap_positions=True):
    type_map = infer_type_map(traj_path, ref_xyz_path)
    frames = []

    for frame_index, frame in enumerate(iter_lammpstrj_frames(traj_path)):
        if frame_index % stride != 0:
            continue

        order = np.argsort(frame["ids"])
        atoms = Atoms(
            symbols=[type_map[int(atom_type)] for atom_type in frame["types"][order]],
            positions=frame["positions"][order],
            cell=frame["cell"],
            pbc=True,
        )

        if wrap_positions:
            atoms.wrap()

        atoms.info["timestep"] = int(frame["timestep"])
        if frame["forces"] is not None:
            atoms.arrays["lammps_forces"] = frame["forces"][order]

        frames.append(atoms)
        if max_frames is not None and len(frames) >= max_frames:
            break

    return frames, type_map


In [8]:
frames, type_map = load_frames(
    TRAJ_PATH,
    REF_PATH,
    stride=STRIDE,
    max_frames=MAX_FRAMES,
    wrap_positions=WRAP_POSITIONS,
)

timesteps = [atoms.info["timestep"] for atoms in frames]
print(f"Loaded {len(frames)} frames")
print(f"Type map: {type_map}")
print(f"Timesteps: {timesteps[0]} -> {timesteps[-1]} (stride in dumped frames: {STRIDE})")
print(frames[0])

Loaded 1000 frames
Type map: {3: 'Ba', 2: 'Ti', 1: 'O'}
Timesteps: 0 -> 99900 (stride in dumped frames: 100)
Atoms(symbols='Ba27O81Ti27', pbc=True, cell=[11.97113763, 11.97113763, 12.30796617], lammps_forces=...)


## Smooth notebook trajectory viewer (`nglview`)

This is the recommended in-notebook viewer for long trajectories. It gives smooth play/pause, frame scrubbing, dragging, and better camera interaction than `py3Dmol`. If `nglview` is not installed, use `pip install nglview`.


In [9]:
from ase.visualize import view
view(frames, viewer="ngl")
